### import env var and libraries

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import json


load_dotenv()

client = OpenAI()

/home/pouya/Documents/AI-Engineer-SDS/ai_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### set up Pushover

In [2]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

#Test pushover
import requests

def send_notification(message: str):
    payload = { "user": pushover_user, "token": pushover_token, "message": message }
    response = requests.post(pushover_url, data=payload)
    return response

#describe pushover as an LLM tool
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a pushover notification to the user's phone via the pushover API. Use this to alert the user about important information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
             }
            },
        "required": ["message"]
        }
}

#add pushover to list of LLM tool
tools = [{"type": "function", "function": send_notification_function}]

### Create new function, describe it, and add it to the list of tools

In [5]:
import random

#simulates rolling a 6-sided dice
def dice_roll():
    result = random.randint(1,6)
    return result

# describe function for LLM
roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulate rolling a single six-sided die and returns the result. Use this when the user wnats to roll a die for games, decisions, or random number generation.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
        }
} 

#Add function to list of tools LLM can use
tools.append({"type": "function", "function": roll_dice_function})

### LLM tool call handling

In [8]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name 
        args = json.loads(tool_call.function.arguments)

        #Route to the appropriate function based on the function_name
        if function_name == "send_notification":
            send_notification(args['message'])
            content = f"Notification sent: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
        # elif function_name == "function_name_v3":
        #      call function_name_v3
        #... 
        else:
            content = f"Unkown Function: {function_name}"


        tool_call_result = {
            "role": "tool" ,
            "content": content, 
            "tool_call_id": tool_call.id
        }
        tool_results.append(tool_call_result)

    #return what to add to our "context" (about tool call results), a dictionary
    return tool_results

### proper tool call handling 

In [19]:
from litellm import completion
client = OpenAI()
messages = [{"role": "user", "content": " roll a dice twice, then send me a noti about it "}]

response = completion(
    model="gpt-4.1-mini",
    messages = messages,
    tools=tools,
    tool_choice="auto"
)
message = response.choices[0].message


#Check if model wants to call a tool
while message.tool_calls:
    # from pprint import pprint
    # pprint(message.tool_calls) #for debugging

    #.. handle tool call
    toolDict = handle_tool_call(message.tool_calls)

    #.. add message to context , i.e messages
    messages.append(message)

    #.. add info about tool call response to message (context)so t
    messages.extend(toolDict)

    #.. invoke LLM again to get its updated response    
    response = completion(
        model="gpt-4.1-mini",
        messages = messages,
        tools=tools,
        tool_choice="auto"
    )
    message = response.choices[0].message
    
    #maybe consider adding protection from infinate loop 

print(message.content)


You rolled the dice twice and got a 2 on the first roll and a 5 on the second roll. I've sent you a notification with this result.
